# CIOT-V2: equal-memory head-to-head

Same prompts and grading as the laptop run (GSM8K exact answer, HumanEval unit tests), thinking pre-closed, greedy.

| model | file | size |
|---|---|---|
| Ternary Bonsai 2 27B | PQ2_0 | 7.21 GB |
| Qwen3.8-27B (same base model, ordinary 2-bit) | UD-IQ2_XXS | 7.27 GB |
| Qwen3-8B | Q6_K | 6.73 GB |

**Before running:** Settings → Accelerator **GPU T4 x2**, Internet **On**. Then **Save Version → Save & Run All (Commit)** so it
runs in the background (about 3 hours). Download `ciot_results.zip` from the Output tab when it finishes.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!nproc; free -g | head -2; df -h /tmp /kaggle/working | tail -2

In [ ]:
%%bash
# PrismML's llama.cpp fork, pinned to the commit the laptop numbers use, built for the T4 (sm_75)
set -e
mkdir -p /tmp/ciot/eval && cd /tmp/ciot
if [ ! -x llama.cpp/build/bin/llama-server ]; then
  git clone --quiet --filter=blob:none -b prism https://github.com/PrismML-Eng/llama.cpp
  cd llama.cpp && git checkout --quiet 6bfcd79a2d426abcd2b50e3c2d09ae2225e70a17
  cmake -B build -DCMAKE_BUILD_TYPE=Release -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75 -DLLAMA_CURL=OFF \
        -DLLAMA_BUILD_SERVER=ON -DLLAMA_BUILD_TESTS=OFF > cmake.log 2>&1 || (tail -40 cmake.log; exit 1)
  cmake --build build --target llama-server -j $(nproc) > build.log 2>&1 || (tail -40 build.log; exit 1)
fi
ls -la /tmp/ciot/llama.cpp/build/bin/llama-server

In [ ]:
%%writefile /tmp/ciot/eval/fetch_data.py
"""Download the eval sets once into bench/eval/data as JSONL (stdlib only).

GSM8K test (openai/gsm8k, main) and HumanEval (openai/openai_humaneval), via the
Hugging Face datasets-server rows API.
"""
import json
import os
import urllib.parse
import urllib.request

HERE = os.path.dirname(os.path.abspath(__file__))
DATA = os.path.join(HERE, "data")


def rows(dataset, config, split, total):
    out = []
    while len(out) < total:
        q = urllib.parse.urlencode({"dataset": dataset, "config": config, "split": split,
                                    "offset": len(out), "length": min(100, total - len(out))})
        with urllib.request.urlopen(f"https://datasets-server.huggingface.co/rows?{q}", timeout=60) as r:
            batch = json.load(r)["rows"]
        if not batch:
            break
        out.extend(b["row"] for b in batch)
    return out


def save(name, items):
    os.makedirs(DATA, exist_ok=True)
    path = os.path.join(DATA, name)
    with open(path, "w", encoding="utf-8", newline="\n") as f:
        for it in items:
            f.write(json.dumps(it, ensure_ascii=False) + "\n")
    print(f"{path}: {len(items)} items")


if __name__ == "__main__":
    save("gsm8k_test.jsonl", rows("openai/gsm8k", "main", "test", 1319))
    save("humaneval.jsonl", rows("openai/openai_humaneval", "openai_humaneval", "test", 164))

In [ ]:
import os
os.makedirs("/tmp/ciot/eval", exist_ok=True)
!python /tmp/ciot/eval/fetch_data.py

In [ ]:
%%writefile /tmp/ciot/eval/run_eval.py
"""Equal-memory head-to-head on this laptop: accuracy and wall-clock time per task.

Starts llama-server once per model, sends the same prompts to every model (ChatML,
thinking pre-closed, greedy), and grades:
  - GSM8K: exact final number on an 'Answer: <number>' line
  - HumanEval: the official unit tests, run in a subprocess (model-written code is
    executed locally in a temp dir with a 30 s timeout; this is not a sandbox)

usage: python bench/eval/run_eval.py --model models/X.gguf --name X [--gsm 60] [--he 40]
"""
import argparse
import json
import os
import re
import subprocess
import sys
import tempfile
import time
import urllib.request

HERE = os.path.dirname(os.path.abspath(__file__))
ROOT = os.path.abspath(os.path.join(HERE, "..", ".."))
SERVER = os.path.join(ROOT, "vendor", "prism-llama.cpp", "build", "bin", "llama-server.exe")
STOP = ["<|im_end|>", "<|endoftext|>"]


def chat(user):
    # thinking pre-closed: the model answers directly, same for every model in the comparison
    return f"<|im_start|>user\n{user}<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"


def load(name, n):
    with open(os.path.join(HERE, "data", name), encoding="utf-8") as f:
        return [json.loads(line) for line in f][:n]


def post(port, payload, timeout=1800):
    req = urllib.request.Request(f"http://127.0.0.1:{port}/completion", data=json.dumps(payload).encode(),
                                 headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.load(r)


def wait_ready(port, proc, timeout=600):
    t0 = time.time()
    while time.time() - t0 < timeout:
        if proc.poll() is not None:
            raise RuntimeError("server exited during load")
        try:
            with urllib.request.urlopen(f"http://127.0.0.1:{port}/health", timeout=5) as r:
                if json.load(r).get("status") == "ok":
                    return time.time() - t0
        except Exception:
            pass
        time.sleep(2)
    raise RuntimeError("server did not become ready")


def last_number(text):
    m = re.findall(r"Answer:\s*\$?\s*(-?[\d,]*\.?\d+)", text)
    if not m:
        m = re.findall(r"(-?[\d,]*\.?\d+)", text)
    if not m:
        return None
    try:
        return float(m[-1].replace(",", ""))
    except ValueError:
        return None


def grade_gsm(item, text):
    gold = float(item["answer"].split("####")[-1].strip().replace(",", ""))
    got = last_number(text)
    return got is not None and abs(got - gold) < 1e-6


def extract_code(text):
    # the closing fence is optional: models often end the turn right after the code
    blocks = re.findall(r"```(?:python|py)?[ \t]*\n(.*?)(?:```|\Z)", text, re.S)
    if blocks:
        return blocks[0]
    return text


def grade_he(item, text):
    code = extract_code(text)
    imports = "\n".join(l for l in item["prompt"].splitlines() if l.startswith(("import ", "from ")))
    if f"def {item['entry_point']}" not in code:
        code = item["prompt"] + code  # model continued the body only
    program = f"{imports}\n\n{code}\n\n{item['test']}\n\ncheck({item['entry_point']})\n"
    with tempfile.TemporaryDirectory() as d:
        path = os.path.join(d, "t.py")
        with open(path, "w", encoding="utf-8") as f:
            f.write(program)
        try:
            r = subprocess.run([sys.executable, path], cwd=d, capture_output=True, timeout=30)
            return r.returncode == 0
        except subprocess.TimeoutExpired:
            return False


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", required=True)
    ap.add_argument("--name", required=True)
    ap.add_argument("--gsm", type=int, default=60)
    ap.add_argument("--he", type=int, default=40)
    ap.add_argument("--port", type=int, default=8091)
    ap.add_argument("--threads", type=int, default=8)
    ap.add_argument("--server-args", default="")
    ap.add_argument("--server", default=SERVER, help="llama-server binary")
    ap.add_argument("--gpu", action="store_true", help="offload all layers to the GPU instead of the laptop CPU settings")
    ap.add_argument("--out", default=os.path.join(HERE, "results"))
    args = ap.parse_args()

    out_dir = args.out
    os.makedirs(out_dir, exist_ok=True)
    log = open(os.path.join(out_dir, f"{args.name}.server.log"), "w", encoding="utf-8")
    if args.gpu:
        device = ["-ngl", "99"]
    else:  # laptop CPU: high priority so Windows does not preempt spinning workers, weights read into RAM
        device = ["-t", str(args.threads), "-tb", str(args.threads), "--prio", "2", "--no-mmap"]
    cmd = [args.server, "-m", args.model] + device + ["-c", "4096", "-np", "1", "--port", str(args.port),
                                                      "--host", "127.0.0.1"] + args.server_args.split()
    proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, cwd=ROOT, env=dict(os.environ, CIOT_T2="1"))
    try:
        load_s = wait_ready(args.port, proc)
        print(f"[{args.name}] loaded in {load_s:.0f}s", flush=True)

        tasks = [("gsm8k", it) for it in load("gsm8k_test.jsonl", args.gsm)] + \
                [("humaneval", it) for it in load("humaneval.jsonl", args.he)]
        results = []
        partial = open(os.path.join(out_dir, f"{args.name}.partial.jsonl"), "w", encoding="utf-8", newline="\n")
        for idx, (kind, item) in enumerate(tasks):
            if kind == "gsm8k":
                prompt = chat(f"{item['question']}\n\nSolve this step by step, briefly. "
                              "End with a final line of the form 'Answer: <number>'.")
                n_predict = 400
            else:
                prompt = chat("Complete this Python function. Reply with only the complete function, including "
                              "its signature and any imports it needs, in one ```python code block.\n\n"
                              f"```python\n{item['prompt']}```")
                n_predict = 512
            t0 = time.perf_counter()
            r = post(args.port, {"prompt": prompt, "n_predict": n_predict, "temperature": 0, "top_k": 1,
                                 "cache_prompt": False, "stop": STOP})
            wall = time.perf_counter() - t0
            text = r.get("content", "")
            ok = grade_gsm(item, text) if kind == "gsm8k" else grade_he(item, text)
            tm = r.get("timings", {})
            rec = {"task": kind, "id": item.get("task_id", idx), "correct": ok, "wall_s": wall,
                   "prompt_n": tm.get("prompt_n"), "prompt_ms": tm.get("prompt_ms"),
                   "gen_n": tm.get("predicted_n"), "gen_ms": tm.get("predicted_ms"),
                   "truncated": r.get("truncated", False) or tm.get("predicted_n", 0) >= n_predict, "text": text}
            results.append(rec)
            partial.write(json.dumps(rec, ensure_ascii=False) + "\n")
            partial.flush()
            print(f"[{args.name}] {idx + 1}/{len(tasks)} {kind} {'ok ' if ok else 'BAD'} {wall:6.1f}s "
                  f"gen {rec['gen_n']} tok", flush=True)

        partial.close()
        with open(os.path.join(out_dir, f"{args.name}.jsonl"), "w", encoding="utf-8", newline="\n") as f:
            for rec in results:
                f.write(json.dumps(rec, ensure_ascii=False) + "\n")

        summary = {"name": args.name, "model": os.path.basename(args.model),
                   "model_gb": os.path.getsize(args.model) / 1e9, "load_s": load_s,
                   "device": "gpu" if args.gpu else "cpu"}
        for kind in ("gsm8k", "humaneval"):
            rs = [x for x in results if x["task"] == kind]
            if not rs:
                continue
            n_ok = sum(x["correct"] for x in rs)
            wall = sum(x["wall_s"] for x in rs)
            gen = sum(x["gen_n"] or 0 for x in rs)
            gen_ms = sum(x["gen_ms"] or 0 for x in rs)
            summary[kind] = {"n": len(rs), "correct": n_ok, "accuracy": n_ok / len(rs), "wall_s": wall,
                             "s_per_task": wall / len(rs), "s_per_correct": wall / max(n_ok, 1),
                             "prompt_tokens": sum(x["prompt_n"] or 0 for x in rs),
                             "gen_tokens": gen, "decode_tok_s": gen / (gen_ms / 1000) if gen_ms else None,
                             "truncated": sum(x["truncated"] for x in rs)}
        with open(os.path.join(out_dir, f"{args.name}.summary.json"), "w", encoding="utf-8") as f:
            json.dump(summary, f, indent=2)
        print(json.dumps(summary, indent=2), flush=True)
    finally:
        proc.terminate()
        try:
            proc.wait(timeout=30)
        except subprocess.TimeoutExpired:
            proc.kill()
        log.close()


if __name__ == "__main__":
    main()

In [ ]:
import os, subprocess, sys
from huggingface_hub import hf_hub_download

GSM, HE = 250, 164   # GSM8K problems (of 1319) and HumanEval problems (all 164)
OUT = "/kaggle/working/results"
SERVER = "/tmp/ciot/llama.cpp/build/bin/llama-server"
MODELS = [
    ("bonsai2-27b-pq2_0",   "prism-ml/Ternary-Bonsai-2-27B-gguf", "Ternary-Bonsai-2-27B-PQ2_0.gguf"),
    ("qwen3.8-27b-iq2_xxs", "unsloth/Qwen3.8-27B-GGUF",           "Qwen3.8-27B-UD-IQ2_XXS.gguf"),
    ("qwen3-8b-q6_k",       "Qwen/Qwen3-8B-GGUF",                 "Qwen3-8B-Q6_K.gguf"),
]
env = dict(os.environ, CUDA_VISIBLE_DEVICES="0")  # every model fits on one T4

for name, repo, fname in MODELS:
    if os.path.exists(f"{OUT}/{name}.summary.json"):
        print(f"{name}: already done"); continue
    path = hf_hub_download(repo, fname, local_dir="/tmp/ciot/models")
    cmd = [sys.executable, "-u", "/tmp/ciot/eval/run_eval.py", "--model", path, "--name", name, "--gsm", str(GSM),
           "--he", str(HE), "--gpu", "--server", SERVER, "--out", OUT]
    with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env) as p:
        for line in p.stdout:
            print(line, end="")
    os.remove(path)  # one model on disk at a time

In [ ]:
import glob, json, shutil
rows = [json.load(open(f)) for f in sorted(glob.glob("/kaggle/working/results/*.summary.json"))]
print("| model | GB | GSM8K | HumanEval | gen tokens per task | T4 decode tok/s |")
print("|---|---|---|---|---|---|")
for r in rows:
    g, h = r.get("gsm8k", {}), r.get("humaneval", {})
    tasks = g.get("n", 0) + h.get("n", 0)
    gen = (g.get("gen_tokens", 0) + h.get("gen_tokens", 0)) / max(tasks, 1)
    print(f"| {r['name']} | {r['model_gb']:.2f} | {g.get('correct')}/{g.get('n')} ({100*g.get('accuracy',0):.1f}%) | "
          f"{h.get('correct')}/{h.get('n')} ({100*h.get('accuracy',0):.1f}%) | {gen:.0f} | {g.get('decode_tok_s') or 0:.1f} |")
shutil.make_archive("/kaggle/working/ciot_results", "zip", "/kaggle/working/results")
print("saved /kaggle/working/ciot_results.zip")